# 🤖 Scikit-Learn Integration Example

Integrate `mlprep` into a Python-based machine learning workflow.

## Workflow
1. **Feature Engineering with mlprep**: Scalable, consistent preprocessing
2. **Training with Scikit-Learn**: Load processed Parquet file for model training

In [ ]:
!pip install -q "mlprep-rust==0.3.0" pandas pyarrow scikit-learn

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Setup output directory
BASE = Path.cwd() / 'outputs'
BASE.mkdir(exist_ok=True)

np.random.seed(42)
n_rows = 200
df = pd.DataFrame({
    'feature1': np.random.normal(0, 1, n_rows),
    'feature2': np.random.normal(5, 2, n_rows),
    'feature3': np.random.choice(['A', 'B'], n_rows),
    'target': np.random.randint(0, 2, n_rows)
})
df.to_csv('raw_data.csv', index=False)
print('Generated raw_data.csv')
df.head()

In [ ]:
pipeline_yaml = '''name: sklearn_prep
inputs:
  - path: raw_data.csv
    format: csv

steps:
  - type: select
    columns: [feature1, feature2, feature3, target]
  - type: features
    config:
      features:
        - column: feature1
          transform: standard_scale
        - column: feature2
          transform: standard_scale

outputs:
  - path: outputs/processed_train.parquet
    format: parquet
'''

with open('pipeline.yaml', 'w') as f:
    f.write(pipeline_yaml)
print(pipeline_yaml)

In [ ]:
!mlprep run pipeline.yaml --streaming --memory-limit 1GB

In [ ]:
import os
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

if os.path.exists('outputs/processed_train.parquet'):
    df = pd.read_parquet('outputs/processed_train.parquet')
    print(f'Shape: {df.shape}, Columns: {df.columns.tolist()}')
    
    X = df[['feature1', 'feature2']]
    y = df['target']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f'Train Accuracy: {train_acc:.4f}')
    print(f'Test Accuracy: {test_acc:.4f}')
else:
    print('Output not found. Run mlprep pipeline first.')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

if 'model' in dir():
    test_preds = model.predict(X_test)
    print(classification_report(y_test, test_preds))
    cm = confusion_matrix(y_test, test_preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.show()